In [1]:
# python
import sys
import os
import importlib
# columnar analysis
from coffea import processor
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema
import awkward as ak
from dask.distributed import Client, performance_report
# local

sidm_path = str(os.getcwd()).split("/sidm")[0]
# sidm_path = str(sys.path[0]).split("/sidm")[0]
if sidm_path not in sys.path: sys.path.insert(1, sidm_path)
from sidm.tools import utilities, sidm_processor, scaleout, cutflow
from sidm.tools import llpnanoaodschema
# always reload local modules to pick up changes during development
importlib.reload(utilities)
importlib.reload(sidm_processor)
importlib.reload(scaleout)
# plotting
import matplotlib.pyplot as plt
utilities.set_plot_style()
%matplotlib inline
from tqdm.notebook import tqdm
import coffea.util
import numpy as np
import mplhep as hep

In [ ]:
import awkward as ak
import json
import os
from collections import defaultdict
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema
import yaml
from datetime import datetime

# Config
yaml_file_path = "/home/cms-jovyan/workspace/CMS-SIDM-PR/sidm/configs/ntuples/data_skimmed.yaml"
output_dir = "/home/cms-jovyan/workspace/CMS-SIDM-PR/sidm/studies/JSON"
log_dir = os.path.join(output_dir, "logs")

os.makedirs(output_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

# Load sample list from YAML
with open(yaml_file_path, "r") as f:
    data = yaml.safe_load(f)

data_all = list(data["llpNanoAOD_v2"]["samples"].keys())

print(f"Total samples in YAML: {len(data_all)}")


def write_log_header(f, sample):
    f.write(f"# Sample: {sample}\n")
    f.write(f"# Created: {datetime.now().isoformat()}\n\n")


# Loop over samples
for sample in data_all:
    print("\n" + "=" * 80)
    print(f"Processing sample: {sample}")

    processed_json = os.path.join(output_dir, f"processed_lumis_{sample}.json")
    filemap_json   = os.path.join(output_dir, f"file_runlumi_map_{sample}.json")
    error_log      = os.path.join(log_dir, f"error_log_{sample}.log")

    is_complete = (
        os.path.exists(processed_json)
        and os.path.exists(filemap_json)
        and os.path.exists(error_log)
    )

    if is_complete:
        print("  -> Skip: processed_json, filemap_json, and error_log already exist")
        print(f"     {processed_json}")
        print(f"     {filemap_json}")
        print(f"     {error_log}")
        continue

    lumi_dict = defaultdict(set)
    file_map = {}

    open_errors = []
    read_errors = []
    empty_files = []

    # Build fileset
    try:
        fileset_data = utilities.make_fileset(
            [sample],
            "llpNanoAOD_v2",
            max_files=-1,
            location_cfg="data_skimmed.yaml",
        )
    except Exception as e:
        print(f"  -> Error making fileset for sample {sample}")
        print(f"     {e}")

        with open(error_log, "w") as f:
            write_log_header(f, sample)
            f.write("[FILESET_ERROR]\n")
            f.write(str(e) + "\n")
        continue

    if sample not in fileset_data or "files" not in fileset_data[sample]:
        print("  -> No files found in fileset, skip")
        with open(error_log, "w") as f:
            write_log_header(f, sample)
            f.write("[STATUS]\n")
            f.write("No files found in fileset.\n")
        continue

    file_list = fileset_data[sample]["files"]
    n_files = len(file_list)

    print(f"  -> Number of files: {n_files}")

    if n_files == 0:
        print("  -> Empty file list, skip")
        with open(error_log, "w") as f:
            write_log_header(f, sample)
            f.write("[STATUS]\n")
            f.write("File list is empty.\n")
        continue

    print_every = 100 if n_files >= 100 else 10

    # Read ROOT files one by one
    for i, file_path in enumerate(file_list, start=1):
        if (i == 1) or (i % print_every == 0) or (i == n_files):
            print(f"     Processing file {i}/{n_files}")

        try:
            events = NanoEventsFactory.from_root(
                {file_path: "Events"},
                schemaclass=NanoAODSchema,
            ).events()
        except Exception as e:
            print(f"       Error opening file: {file_path}")
            print(f"       {e}")
            open_errors.append({
                "file": file_path,
                "error": str(e),
            })
            continue

        try:
            runs = ak.to_numpy(events.run)
            lumis = ak.to_numpy(events.luminosityBlock)
        except Exception as e:
            print(f"       Error reading run/lumi branches: {file_path}")
            print(f"       {e}")
            read_errors.append({
                "file": file_path,
                "error": str(e),
            })
            continue

        if len(runs) == 0 or len(lumis) == 0:
            print(f"       Empty file, skip: {file_path}")
            empty_files.append(file_path)
            continue

        file_pairs = set()

        for r, l in zip(runs, lumis):
            r = int(r)
            l = int(l)
            lumi_dict[r].add(l)
            file_pairs.add((r, l))

        file_map[file_path] = sorted(file_pairs)

    # Convert to CMS-style JSON
    cms_json = {}

    for run, lumis in lumi_dict.items():
        lumis = sorted(lumis)

        if len(lumis) == 0:
            continue

        ranges = []
        start = lumis[0]
        prev = lumis[0]

        for l in lumis[1:]:
            if l == prev + 1:
                prev = l
            else:
                ranges.append([start, prev])
                start = l
                prev = l

        ranges.append([start, prev])
        cms_json[str(run)] = ranges

    # Save outputs
    save_error = None
    try:
        with open(processed_json, "w") as f:
            json.dump(cms_json, f, indent=2)

        with open(filemap_json, "w") as f:
            json.dump(file_map, f, indent=2)

        print("  -> Saved output files")
        print(f"     {processed_json}")
        print(f"     {filemap_json}")

    except Exception as e:
        print("  -> Error writing output files")
        print(f"     {e}")
        save_error = str(e)

    with open(error_log, "w") as f:
        write_log_header(f, sample)

        f.write("[SUMMARY]\n")
        f.write(f"Total input files      : {n_files}\n")
        f.write(f"Files successfully read: {len(file_map)}\n")
        f.write(f"Open errors            : {len(open_errors)}\n")
        f.write(f"Read errors            : {len(read_errors)}\n")
        f.write(f"Empty files            : {len(empty_files)}\n")
        f.write(f"Runs collected         : {len(cms_json)}\n")
        f.write(f"Processed JSON         : {processed_json}\n")
        f.write(f"FileMap JSON           : {filemap_json}\n")
        f.write("\n")

        if save_error is not None:
            f.write("[SAVE_ERROR]\n")
            f.write(save_error + "\n\n")

        f.write("[OPEN_ERRORS]\n")
        if open_errors:
            for item in open_errors:
                f.write(f"{item['file']}\n")
                f.write(f"  ERROR: {item['error']}\n")
        else:
            f.write("None\n")
        f.write("\n")

        f.write("[READ_ERRORS]\n")
        if read_errors:
            for item in read_errors:
                f.write(f"{item['file']}\n")
                f.write(f"  ERROR: {item['error']}\n")
        else:
            f.write("None\n")
        f.write("\n")

        f.write("[EMPTY_FILES]\n")
        if empty_files:
            for path in empty_files:
                f.write(path + "\n")
        else:
            f.write("None\n")
        f.write("\n")

    print("  -> Saved error log")
    print(f"     {error_log}")

print("\nDone.")